In [1]:
!pip install -q neo4j

In [2]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 134.2 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate]


In [3]:
import torch
import json
from neo4j import GraphDatabase

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

print("Qwen loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!


In [5]:
import neo4j
from getpass import getpass
from neo4j import GraphDatabase

NEO4J_URI =  "neo4j+s://2b1f2d19.databases.neo4j.io"
NEO4J_USERNAME = "2b1f2d19"
NEO4J_PASSWORD = "dH--Ad36MBlpKWcaOkezZCCn_2Angf6qH90RjgPMS8U"
NEO4J_DATABASE = "2b1f2d19"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


In [6]:
schema_query = """
CALL db.schema.visualization()
"""

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(schema_query)
    schema_records = list(result)

print("Schema retrieved.")

Schema retrieved.


In [7]:
with driver.session(database=NEO4J_DATABASE) as session:

    node_result = session.run("""
        MATCH (n)
        RETURN DISTINCT labels(n) AS labels
    """)

    rel_result = session.run("""
        MATCH ()-[r]->()
        RETURN DISTINCT type(r) AS relationship
    """)

    node_labels = [record["labels"] for record in node_result]
    relationship_types = [
        record["relationship"] for record in rel_result
    ]

print("Node labels:", node_labels)
print("Relationship types:", relationship_types)

Node labels: [['Faculty'], ['Department'], ['ResearchArea'], ['Application'], ['Course']]
Relationship types: ['WORKS_IN', 'RESEARCHES', 'TEACHES', 'APPLIED_IN']


In [8]:
sample_query = """
MATCH (n)
RETURN labels(n) AS label, properties(n) AS properties
LIMIT 15
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(sample_query)

    for record in records:
        print(record["label"], ":", record["properties"])

['Faculty'] : {'name': 'Dr. Anu'}
['Department'] : {'name': 'CSE'}
['ResearchArea'] : {'name': 'Knowledge Graphs'}
['Application'] : {'name': 'Bioinformatics'}
['Course'] : {'name': 'Artificial Intelligence'}
['Faculty'] : {'name': 'Dr. Ravi'}
['Department'] : {'name': 'ECE'}
['ResearchArea'] : {'name': 'Machine Learning'}
['Application'] : {'name': 'Healthcare'}
['Course'] : {'name': 'Machine Learning'}
['Faculty'] : {'name': 'Dr. Manu'}
['ResearchArea'] : {'name': 'Bioinformatics'}
['Application'] : {'name': 'Drug Discovery'}
['Course'] : {'name': 'Data Science'}
['Course'] : {'name': 'Knowledge Representation'}


In [9]:
graph_schema = """
Knowledge Graph Schema:

Node labels and properties:

1. Faculty
   - name: string

2. Department
   - name: string

3. ResearchArea
   - name: string

4. Application
   - name: string

5. Course
   - name: string

Relationships:

1. (Faculty)-[:WORKS_IN]->(Department)

2. (Faculty)-[:RESEARCHES]->(ResearchArea)

3. (ResearchArea)-[:APPLIED_IN]->(Application)

4. (Faculty)-[:TEACHES]->(Course)
"""

In [10]:
def ask_qwen(prompt, max_tokens=300):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [11]:
print(ask_qwen("What is a knowledge graph?", 100))

A knowledge graph is a type of graph that represents entities and their relationships in the form of nodes and edges. In the context of knowledge graphs, nodes typically represent entities such as people, places, organizations, or things, while edges represent the relationships between these entities.

For example:
- Nodes: "John", "Alibaba", "Cloud Computing"
- Edges: "is a", "works for", "uses"

In a knowledge graph, you can also have additional metadata associated with nodes and


In [12]:
def generate_cypher(question):

    prompt = f"""
You are an expert in Neo4j Cypher query generation.

Given the following Knowledge Graph schema:

{graph_schema}

Convert the user's question into a Cypher query.

Rules:
1. Generate only a read-only Cypher query.
2. Use only the node labels, relationships and properties
   defined in the schema.
3. Use MATCH and RETURN.
4. Do not use CREATE, DELETE, SET, MERGE, DROP or other
   database modification commands.
5. Return only the Cypher query.
6. Do not include markdown or explanations.

User Question:
{question}

Cypher Query:
"""

    cypher = ask_qwen(prompt, max_tokens=200)

    # Remove possible markdown formatting
    cypher = cypher.replace("```cypher", "").replace("```", "").strip()

    return cypher

In [16]:
question = "Which faculty members are research in Bioinformatics?"

cypher_query = generate_cypher(question)

print("Generated Cypher:")
print(cypher_query)

Generated Cypher:
MATCH (f:Faculty)-[:RESEARCHES]->(ra:ResearchArea)-[:APPLIED_IN]->(a:Application {name: 'Bioinformatics'}) RETURN f


In [17]:
import re

def execute_cypher(query):

    # Basic read-only validation
    if not re.match(r"^\s*(MATCH|OPTIONAL\s+MATCH|WITH)\b",
                    query, re.IGNORECASE):
        raise ValueError("Only read-only MATCH/WITH queries are allowed.")

    forbidden = [
        "CREATE", "DELETE", "DETACH", "SET", "REMOVE",
        "MERGE", "DROP", "CALL", "LOAD", "FOREACH"
    ]

    for keyword in forbidden:
        if re.search(r"\b" + keyword + r"\b", query, re.IGNORECASE):
            raise ValueError(f"Forbidden Cypher keyword: {keyword}")

    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.execute_read(
            lambda tx: list(tx.run(query))
        )

    return [record.data() for record in result]

In [18]:
graph_results = execute_cypher(cypher_query)

print("Retrieved Graph Information:")
for item in graph_results:
    print(item)

Retrieved Graph Information:
{'f': {'name': 'Dr. Anu'}}


In [19]:
def generate_answer(question, graph_results):

    prompt = f"""
You are an intelligent assistant answering questions
using a Knowledge Graph.

User Question:
{question}

Retrieved information from the Knowledge Graph:
{json.dumps(graph_results, indent=2)}

Instructions:
1. Answer the user's question using the retrieved information.
2. Do not invent facts.
3. If the retrieved information is insufficient, clearly say so.
4. Provide a concise and understandable answer.

Answer:
"""

    return ask_qwen(prompt, max_tokens=300)

In [20]:
answer = generate_answer(question, graph_results)

print("User Question:", question)
print("\nGraph Results:", graph_results)
print("\nFinal Answer:", answer)

User Question: Which faculty members are research in Bioinformatics?

Graph Results: [{'f': {'name': 'Dr. Anu'}}]

Final Answer: Based on the retrieved information, there is no faculty member named Dr. Anu who is researching in Bioinformatics. The provided data does not contain any faculty members associated with research in Bioinformatics.


In [19]:
def graph_rag(question):

    print("=" * 60)
    print("USER QUESTION:", question)

    # 1. Generate Cypher
    cypher = generate_cypher(question)

    print("\nGENERATED CYPHER:")
    print(cypher)

    # 2. Retrieve information from Neo4j
    results = execute_cypher(cypher)

    print("\nRETRIEVED GRAPH CONTEXT:")
    print(results)

    if not results:
        return "No relevant information was found in the Knowledge Graph."

    # 3. Generate answer
    answer = generate_answer(question, results)

    print("\nFINAL ANSWER:")
    print(answer)

    print("=" * 60)

    return answer

In [20]:
graph_rag("Which faculty members are working on NLP?")

USER QUESTION: Which faculty members are working on NLP?

GENERATED CYPHER:
MATCH (f:Faculty)-[:WORKS_IN]->(d:Department)-[:APPLIED_IN]->(a:Application {name: 'NLP'}) RETURN f.name,f.works_in,d.name,a.name



RETRIEVED GRAPH CONTEXT:
[]


'No relevant information was found in the Knowledge Graph.'